In [2]:
!pip install pandas numpy scikit-learn xgboost openpyxl joblib


   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/250.9 kB 660.6 kB/s eta 0:00:01
   ------ -------------------------------- 41.0/250.9 kB 495.5 kB/s eta 0:00:01
   --------- ----------------------------- 61.4/250.9 kB 469.7 kB/s eta 0:00:01
   -------------- ------------------------ 92.2/250.9 kB 479.1 kB/s eta 0:00:01
   ----------------- -------------------- 112.6/250.9 kB 504.4 kB/s eta 0:00:01
   ------------------ ------------------- 122.9/250.9 kB 481.4 kB/s eta 0:00:01
   ----------------------- -------------- 153.6/250.9 kB 459.5 kB/s eta 0:00:01
   -------------------------- ----------- 174.1/250.9 kB 477.7 kB/s eta 0:00:01
   -------------------------- ----------- 174.1/250.9 kB 477.7 kB/s eta 0:00:01
   -------------------------- ----------- 174.1/250.9 kB 477.7 kB/s eta 0:00:01
   -------------------------- ----------- 174.1/250.9 kB 


[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: C:\Users\DELL\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np


In [7]:
file_path = "health_nutrition_disease_dataset_12000.xlsx"
df = pd.read_excel(file_path)

print(df.shape)
df.head()


(12000, 26)


,Age,Gender,BMI,Daily_Calories_kcal,Carbohydrates_g,Protein_g,Total_Fat_g,Saturated_Fat_g,Trans_Fat_g,Total_Sugar_g,...,Vitamin_D_IU,Vitamin_B12_mcg,Physical_Activity_min,Water_Intake_L,Diabetes_Risk,Hypertension_Risk,Heart_Disease_Risk,Obesity_Risk,Anemia_Risk,Kidney_Disease_Risk
0,65,Female,31.9,2723,256,90,58,68,3.74,85,...,861,4.21,78,2.64,High,High,Low,Low,Low,Low
1,22,Male,40.7,2315,311,131,138,24,4.43,79,...,348,1.32,77,1.04,High,Low,Low,Low,Low,High
2,43,Female,20.8,2519,206,201,166,20,2.67,206,...,199,2.16,163,2.17,Low,Low,Low,Low,Low,Low
3,72,Female,19.6,3536,228,71,52,48,2.79,5,...,1001,3.61,191,3.85,Low,Low,Low,Low,Low,Low
4,21,Female,26.2,2285,257,96,45,57,4.17,54,...,19,2.66,89,3.48,Low,Low,Low,Low,Low,Low


In [8]:
# Gender encoding
df["Gender"] = df["Gender"].map({"Male": 0, "Female": 1})

# Disease columns
disease_columns = [
    "Diabetes_Risk",
    "Hypertension_Risk",
    "Heart_Disease_Risk",
    "Obesity_Risk",
    "Anemia_Risk",
    "Kidney_Disease_Risk"
]

# Convert High / Low → 1 / 0
for col in disease_columns:
    df[col] = df[col].map({"High": 1, "Low": 0})



In [9]:
features = [
    "Age", "Gender", "BMI",
    "Daily_Calories_kcal",
    "Carbohydrates_g",
    "Protein_g",
    "Total_Fat_g",
    "Saturated_Fat_g",
    "Trans_Fat_g",
    "Total_Sugar_g",
    "Added_Sugar_g",
    "Fiber_g",
    "Sodium_mg",
    "Potassium_mg",
    "Calcium_mg",
    "Iron_mg",
    "Vitamin_D_IU",
    "Vitamin_B12_mcg",
    "Physical_Activity_min",
    "Water_Intake_L"
]

X = df[features]


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [11]:
X_train, X_test = train_test_split(
    X, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
import joblib


In [13]:
models = {}

for disease in disease_columns:
    y = df[disease]

    y_train, y_test = train_test_split(
        y, test_size=0.2, random_state=42
    )

    model = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42
    )

    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, preds)
    print(f"{disease} accuracy: {acc:.3f}")

    models[disease] = model


Diabetes_Risk accuracy: 0.999
Hypertension_Risk accuracy: 0.999
Heart_Disease_Risk accuracy: 1.000
Obesity_Risk accuracy: 0.999
Anemia_Risk accuracy: 1.000
Kidney_Disease_Risk accuracy: 1.000


In [14]:
joblib.dump(scaler, "scaler.pkl")

for disease, model in models.items():
    joblib.dump(model, f"{disease}_model.pkl")

print("Models saved successfully")


Models saved successfully


In [15]:
def calculate_bmi(weight_kg, height_cm):
    height_m = height_cm / 100
    return round(weight_kg / (height_m ** 2), 1)


In [16]:
patient = {
    "Age": 45,
    "Gender": 1,  # 0 = Male, 1 = Female
    "Weight_kg": 70,
    "Height_cm": 154,
    "Daily_Calories_kcal": 2800,
    "Carbohydrates_g": 350,
    "Protein_g": 90,
    "Total_Fat_g": 130,
    "Saturated_Fat_g": 45,
    "Trans_Fat_g": 2,
    "Total_Sugar_g": 120,
    "Added_Sugar_g": 90,
    "Fiber_g": 15,
    "Sodium_mg": 3500,
    "Potassium_mg": 2000,
    "Calcium_mg": 500,
    "Iron_mg": 7,
    "Vitamin_D_IU": 2190,
    "Vitamin_B12_mcg": 1.8,
    "Physical_Activity_min": 20,
    "Water_Intake_L": 1.2
}


In [17]:
patient["BMI"] = calculate_bmi(
    patient["Weight_kg"], patient["Height_cm"]
)

patient_df = pd.DataFrame([{
    k: patient[k] for k in features
}])

patient_scaled = scaler.transform(patient_df)


In [19]:
print("Disease Risk Prediction  \n")

for disease in disease_columns:
    model = models[disease]
    prediction = model.predict(patient_scaled)[0]
    risk = "High Risk" if prediction == 1 else "Low Risk"
    print(f"{disease.replace('_', ' ')}: {risk}")


Disease Risk Prediction

Diabetes Risk: Low Risk
Hypertension Risk: High Risk
Heart Disease Risk: High Risk
Obesity Risk: Low Risk
Anemia Risk: High Risk
Kidney Disease Risk: High Risk
